<a href="https://colab.research.google.com/github/rymadinari/-arene-des-algos-Ryma-Dinari-/blob/main/Jour_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# PHASE A : Régression — California Housing
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

def charger_immobilier():
    data = fetch_california_housing()
    X, y = data.data, data.target
    print(f"California Housing : {X.shape}, cible = prix médian en centaines de milliers de $")
    print(f"Variables : {data.feature_names}")
    return X, y

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return {
        "r2"  : r2_score(y_test, y_pred),
        "mae" : mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

X, y = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

California Housing : (20640, 8), cible = prix médian en centaines de milliers de $
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
LinearRegression     R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         R2=0.81  MAE=0.33  RMSE=0.51


In [7]:
# CHECKPOINTS PHASE A

# Checkpoint 1 : cas normal
print("CHECKPOINT 1 — Dataset complet")
for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

print()

# Checkpoint 2 : cas limite (100 lignes seulement)
print("CHECKPOINT 2 — Seulement 100 lignes d'entraînement")
X_100 = X_train_s[:100]
y_100 = y_train[:100]

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_100, X_test_s, y_100, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

print("""
Observation : le R2 s'effondre (peut devenir négatif sur le Random Forest).
Parceque Avec 100 lignes sur 20 640, le modèle n'a pas vu assez de cas
pour généraliser. Le Random Forest surfit : il mémorise les 100 exemples
au lieu d'apprendre une vraie règle. La régression linéaire résiste mieux
car elle a moins de paramètres à estimer.
""")
print()

# Checkpoint 3 : cas adversarial
print("CHECKPOINT 3 — Quartier fictif hors plage")

lr_final = LinearRegression()
lr_final.fit(X_train_s, y_train)


quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 37.0, -120.0]])

quartier_fictif_s = scaler.transform(quartier_fictif)

pred_lr = lr_final.predict(quartier_fictif_s)[0]

rf_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_final.fit(X_train_s, y_train)
pred_rf = rf_final.predict(quartier_fictif_s)[0]

print(f"Quartier fictif : revenu=0, population=9000")
print(f"LinearRegression → prix prédit : {pred_lr:.2f} (x100k$)")
print(f"RandomForest     → prix prédit : {pred_rf:.2f} (x100k$)")

if pred_lr < 0:
    print(f"  Régression linéaire : prix NÉGATIF ({pred_lr:.2f}) — valeur absurde !")
else:
    print(f" Valeur hors plage des données d'entraînement — à surveiller")

print(f"Plage normale des prix dans le dataset : "
      f"{y.min():.2f} à {y.max():.2f} (x100k$)")

print("""
Que faire en production ?
1. Valider les entrées AVANT le modèle : rejeter tout revenu < 0
   ou toute population hors de la plage vue à l'entraînement.
2. Ajouter une couche de détection d'anomalies en amont.
3. Ne jamais faire confiance à une prédiction sur une entrée
   que le modèle n'a jamais vue : extrapoler, c'est risqué.
""")

CHECKPOINT 1 — Dataset complet
LinearRegression     R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         R2=0.81  MAE=0.33  RMSE=0.51

CHECKPOINT 2 — Seulement 100 lignes d'entraînement
LinearRegression     R2=0.40  MAE=0.54  RMSE=0.88
RandomForest         R2=0.54  MAE=0.56  RMSE=0.77

Observation : le R2 s'effondre (peut devenir négatif sur le Random Forest).
Parceque Avec 100 lignes sur 20 640, le modèle n'a pas vu assez de cas
pour généraliser. Le Random Forest surfit : il mémorise les 100 exemples
au lieu d'apprendre une vraie règle. La régression linéaire résiste mieux
car elle a moins de paramètres à estimer.


CHECKPOINT 3 — Quartier fictif hors plage
Quartier fictif : revenu=0, population=9000
LinearRegression → prix prédit : -0.18 (x100k$)
RandomForest     → prix prédit : 0.71 (x100k$)
  Régression linéaire : prix NÉGATIF (-0.18) — valeur absurde !
Plage normale des prix dans le dataset : 0.15 à 5.00 (x100k$)

Que faire en production ?
1. Valider les entrées AVANT le modèle : rej

In [8]:
 # PHASE B : Clustering — AirBnB

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def charger_airbnb(url_csv):
    df = pd.read_csv(url_csv)
    cols = ["price", "minimum_nights", "number_of_reviews", "availability_365"]
    df = df[cols].copy()

    if df["price"].dtype == "object":
        df["price"] = df["price"].str.replace("[$,]", "", regex=True).astype(float)

    df = df.dropna()
    print(f"Listings chargés : {df.shape[0]} lignes, {df.shape[1]} colonnes retenues")
    return df

def choisir_k(X_scaled, k_range=range(2, 9)):
    print(f"{'k':<5} {'Inertie':>10} {'Silhouette':>12}")
    print("-" * 30)
    meilleur_k, meilleur_score = 2, -1
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X_scaled)
        inertie  = km.inertia_
        silhouette = silhouette_score(X_scaled, labels)
        print(f"{k:<5} {inertie:>10.0f} {silhouette:>12.2f}")
        if silhouette > meilleur_score:
            meilleur_score = silhouette
            meilleur_k = k
    print(f"Segment retenu : k={meilleur_k} (meilleure silhouette)")
    return meilleur_k


url = "https://data.insideairbnb.com/france/ile-de-france/paris/2024-06-10/visualisations/listings.csv"
df_airbnb = charger_airbnb(url)

scaler_ab = StandardScaler()
X_ab = scaler_ab.fit_transform(df_airbnb)

k_optimal = choisir_k(X_ab)

km_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_airbnb["segment"] = km_final.fit_predict(X_ab)
print("Description des segments :")
print(df_airbnb.groupby("segment").mean().round(1))

Listings chargés : 74579 lignes, 4 colonnes retenues
k        Inertie   Silhouette
------------------------------
2         230941         0.81
3         173481         0.44
4         133196         0.47
5          94984         0.48
6          83293         0.49
7          73579         0.50
8          63797         0.38
Segment retenu : k=2 (meilleure silhouette)
Description des segments :
         price  minimum_nights  number_of_reviews  availability_365
segment                                                            
0        290.0             6.1               22.2             157.3
1        202.5           360.4               16.5             304.7


In [9]:
#Cas limite : SANS standardiser
print("CAS LIMITE — KMeans SANS standardisation")
X_brut = df_airbnb[["price","minimum_nights","number_of_reviews","availability_365"]].values

km_brut = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_airbnb["segment_brut"] = km_brut.fit_predict(X_brut)

print("Taille des segments SANS scaling :")
print(df_airbnb["segment_brut"].value_counts())
print(df_airbnb.groupby("segment_brut")[["price","minimum_nights"]].mean().round(1))
print("""
Observation : la colonne 'price' (en centaines) écrase toutes les autres.
Les clusters se forment uniquement sur le prix, les autres variables
sont ignorées. Le clustering ne fait plus que trier par prix.
→ Sans standardiser, KMeans est inutile sur des colonnes d'échelles différentes.
""")


#Cas adversarial : annonce à 100 000€ la nuit
print("CAS ADVERSARIAL — Valeur aberrante 100 000€/nuit")

df_piege = df_airbnb[["price","minimum_nights","number_of_reviews","availability_365"]].copy()

outlier = pd.DataFrame([[100000, 1, 0, 365]],
                        columns=df_piege.columns)
df_piege = pd.concat([df_piege, outlier], ignore_index=True)

scaler_piege = StandardScaler()
X_piege = scaler_piege.fit_transform(df_piege)

km_piege = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
km_piege.fit(X_piege)

print("Centres des clusters AVEC l'outlier :")
print(pd.DataFrame(km_piege.cluster_centers_,
      columns=df_piege.columns).round(2))
print("""
Observation : l'outlier à 100 000€ attire un cluster entier vers lui.
Les autres segments sont déformés pour "s'éloigner" de ce point extrême.
→ Un seul outlier non nettoyé ruine toute la segmentation.
→ C'est exactement pour ça que le nettoyage J2 est un prérequis absolu.
""")

CAS LIMITE — KMeans SANS standardisation
Taille des segments SANS scaling :
segment_brut
0    74356
1      223
Name: count, dtype: int64
               price  minimum_nights
segment_brut                        
0              262.1            10.0
1             9275.9             9.0

Observation : la colonne 'price' (en centaines) écrase toutes les autres.
Les clusters se forment uniquement sur le prix, les autres variables
sont ignorées. Le clustering ne fait plus que trier par prix.
→ Sans standardiser, KMeans est inutile sur des colonnes d'échelles différentes.

CAS ADVERSARIAL — Valeur aberrante 100 000€/nuit
Centres des clusters AVEC l'outlier :
   price  minimum_nights  number_of_reviews  availability_365
0   0.00            -0.1                0.0             -0.01
1  -0.11             9.0               -0.1              1.28

Observation : l'outlier à 100 000€ attire un cluster entier vers lui.
Les autres segments sont déformés pour "s'éloigner" de ce point extrême.
→ Un seul 